## Imports

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [7]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [8]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Methodology
### Objective: 
- Introduce a pricing buffer of -5 to +5 at the base and observe how it helps in reducing price shock, thereby reducing dormancy.
- The Primary metric that are measured (% routeDiff, % Fare-shock, Customer Retention)
- The Secondary metric that are measured (Impact on GMV, RPC)

### Segmentation Criteria: 
- Customer Entry Criteria (Jaipur <> Link) 
- Customer Metrics (Fare-shock history <> LTR <> FE <> RR <> NET <> Segment <> FE2RR)
- Customer Service Affinity (LINK_ONLY <> BOTH <> !AUTO_ONLY Customer) 
- Optional Demographics info (Gender)
- Behavior (frequency of rides)

### Collect Data: 
- fare_estimates_enriched <> order_logs_snapshot <> iallocator_customer_segments
- Create a customer-level aggregated view.

### Stratified Random Sampling: 
- Use randomization to assign customers within each segment to either the control group (A) or the treatment group (B).
- <b>Balance Groups:</b>     
    - Ensure that the size of the groups is balanced. 
    - Conduct a pre-analysis to check if the groups are indeed homogeneous before starting the experiment.
    - This helps in confirming that the segmentation strategy is effective.
- <b>Exclude Outliers:</b>
    - Identify and exclude outliers that might skew the results. 
    - Outliers could be customers with extremely high or low values in the chosen criteria that are not representative of the majority.
	

### Implement the Experiment: 
- Implement the changes for the treatment group and keep the control group unchanged. 
	
### Monitor and Analyse: 
- Monitor the experiment in real-time and, once completed, analyse the results to determine if there are significant differences between the groups.

## Parameter

In [9]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/rapido/analytics/customer/Fare-shock/price_buffer_exp/user_selector


In [10]:
local_datasets = '/Users/rapido/local-datasets/customer/fare-shock/price_buffer_exp/'

In [11]:
## Parameter 
start_date = '20240101'
end_date = '20240131'
city = 'Jaipur'
service = 'Link'

In [12]:
## base

service_mapping = f"""

    SELECT 
        city_display_name city,
        service_level service_name,
        service_detail_id
    FROM 
        datasets.service_mapping
    WHERE 
        city_display_name = '{city}'
        AND service_level = '{service}'
        
"""

df_service_mapping = pd.read_sql(service_mapping, connection)
df_service_mapping

,city,service_name,service_detail_id
0,Jaipur,Link,5ba1f6d0be5b826084cc7202


In [13]:
service_detail_id = df_service_mapping['service_detail_id'].loc[0]
service_detail_id

'5ba1f6d0be5b826084cc7202'

## Datasets

### Fetching data

In [56]:
##customer_funnel

customer_funnel = f"""

    WITH fe_tbl AS (

        SELECT
            DATE_FORMAT(DATE_PARSE(fee.yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            fee.city,
            fee.service_name,
            user_id as user_id,
            fee.fare_estimate_id,
            COALESCE(price_increase_strategy,'no_surge') AS strategy,
            sfe.session_service_level,
            sfe.session_id
        FROM
            hive.pricing.fare_estimates_enriched fee
        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fee.yyyymmdd
            AND sfe.fare_estimate_id = fee.fare_estimate_id
            AND sfe.service_name = fee.service_name
        WHERE 
            fee.yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND fee.service_detail_id = '{service_detail_id}'
    ),

    rr_net_tbl as (
    
        SELECT   
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city_name AS city,
            service_obj_service_name AS service_name,
            customer_id,
            customer_obj_mobile customer_mobile,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            fare_recalculated_reason fare_recal_reason
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND service_detail_id = '{service_detail_id}'
    ),

    fe_merged AS (
        SELECT
            fe_tbl.*,
            rr_net_tbl.customer_id,
            rr_net_tbl.order_id,
            CAST(rr_net_tbl.customer_mobile AS VARCHAR) customer_mobile,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.fare_recal_reason
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.order_date = rr_net_tbl.order_date
            AND fe_tbl.user_id = rr_net_tbl.customer_id
            AND fe_tbl.fare_estimate_id = rr_net_tbl.fare_estimate_id
    )

    SELECT
        user_id,
        MIN(customer_mobile) customer_mobile,
        COUNT(distinct fare_estimate_id) AS fe_count,
        COUNT(distinct order_id) AS rr_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,
        COUNT(DISTINCT session_id) fe_session,
        COUNT(DISTINCT CASE WHEN order_id IS NOT NULL THEN session_id END) gross_sessions,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN session_id END) net_sessions,
        COUNT(DISTINCT CASE WHEN fare_recal_reason IS NOT NULL THEN order_id END) fare_shock_count,
        COUNT(DISTINCT CASE WHEN fare_recal_reason = 'routeDiff' THEN order_id END) jan_routeDiff_count
    FROM 
        fe_merged
    WHERE
        (session_service_level = 'All' OR session_service_level = '{service}')
    GROUP BY 1
"""

df_customer_funnel = pd.read_sql(customer_funnel, connection)

df_customer_funnel.to_csv(local_datasets + 'raw/customer_funnel_{}_{}_{}_to_{}.csv'
                          .format(city, service, start_date, end_date), index=False
                         )
df_customer_funnel

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,fare_shock_count,jan_routeDiff_count
0,5f43af7ad358d132cc510666,None,3,0,0,1,0,0,0,0
1,652aead0fdae147e7a25280c,7023703875,31,3,2,19,3,2,1,1
2,634f9aad7b3bba469e71445e,7597199064,10,2,2,4,2,2,2,2
3,633c71cbf5442130e2036e41,None,1,0,0,1,0,0,0,0
4,65a6354fdfa89c6adefdd4bc,None,1,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
606868,652ff64379a8cf5d8d0fbef2,None,3,0,0,2,0,0,0,0
606869,619876da3f25c368c1e20ed3,None,7,0,0,4,0,0,0,0
606870,647ac2363294841b46de9c01,8529164075,6,1,1,4,1,1,1,1
606871,6483353fa9fd6725f5b33b46,8777392457,12,4,1,3,3,1,0,0


In [72]:
##customer_segment

customer_segment = f"""

    WITH fe_tbl AS (

        SELECT
            user_id
        FROM
            pricing.fare_estimates_enriched fee
        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fee.yyyymmdd
            AND sfe.fare_estimate_id = fee.fare_estimate_id
            AND sfe.service_name = fee.service_name    
        WHERE 
            fee.yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND fee.service_detail_id = '{service_detail_id}'
            AND (session_service_level = 'All' OR session_service_level = '{service}')
        GROUP BY 1
    )

        SELECT
            user_id,
            COALESCE(gender, 'UNKNOWN') gender,
            ltr_jan1,
            ltr_jan31,
            jan_net_rides,
            COALESCE(link_lifetime_stage_jan1, 'NA') link_lifetime_stage_jan1,
            COALESCE(link_lifetime_stage_jan31, 'NA') link_lifetime_stage_jan31,
            COALESCE(recency_segment_jan1, 'NA') recency_segment_jan1,
            COALESCE(recency_segment_jan31, 'NA') recency_segment_jan31,
            COALESCE(service_affinity_jan1, 'NA') service_affinity_jan1,
            COALESCE(service_affinity_jan31, 'NA') service_affinity_jan31
        FROM
        (    
            SELECT 
                user_id,
                MIN(gender_tag) gender,
                MIN(link_lifetime_rides) ltr_jan1,
                MAX(link_lifetime_rides) ltr_jan31,
                MAX(link_lifetime_rides) - MIN(link_lifetime_rides) jan_net_rides,
                MAX(CASE WHEN run_date = '2024-02-18' THEN link_lifetime_stage END) link_lifetime_stage_jan1,
                MAX(CASE WHEN run_date = '2024-02-18' THEN link_lifetime_stage END) link_lifetime_stage_jan31,
                MAX(CASE WHEN run_date = '2024-02-18' THEN link_recency_segment END) recency_segment_jan1,
                MAX(CASE WHEN run_date = '2024-02-18' THEN link_recency_segment END) recency_segment_jan31,
                MAX(CASE WHEN run_date = '2024-02-18' THEN taxi_service_affinity END) service_affinity_jan1,
                MAX(CASE WHEN run_date = '2024-02-18' THEN taxi_service_affinity END) service_affinity_jan31
            FROM 
                fe_tbl
            LEFT JOIN 
                datasets.iallocator_customer_segments
                ON user_id = customer_id 
                -- AND (run_date = '2024-01-01' OR run_date = '2024-01-31') 
                AND run_date = '2024-02-18'
            GROUP BY 1
        )

"""

df_customer_segment = pd.read_sql(customer_segment, connection)

##Local copy
df_customer_segment.to_csv(local_datasets + 'raw/customer_segment_{}_{}_{}_to_{}.csv' 
                           .format(city, service, start_date, end_date), index=False 
                          )
df_customer_segment

,user_id,gender,ltr_jan1,ltr_jan31,jan_net_rides,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31
0,648c29c07f3932084274f699,MALE,66.0,66.0,0.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK
1,63d0085d319e74dff8c961e1,MALE,47.0,47.0,0.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK
2,5c41ee6a4a267149c773930c,MALE,79.0,79.0,0.0,HOOK,HOOK,RECENT,RECENT,ONLY_LINK,ONLY_LINK
3,5d3a8ab39996733edb01df18,MALE,102.0,102.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_AUTO,ONLY_AUTO
4,5bae3c8c1947b01fba73e920,MALE,16.0,16.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,ONLY_AUTO,ONLY_AUTO
...,...,...,...,...,...,...,...,...,...,...,...
606868,630aacc781735c0d926d2169,MALE,4.0,4.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
606869,5f758ef6a649a034b94745ad,FEMALE,107.0,107.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,ONLY_AUTO,ONLY_AUTO
606870,5dd103e20856f14618344314,FEMALE,3.0,3.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
606871,6553b39be10b6e5f929d9766,UNKNOWN,NaN,NaN,NaN,NA,NA,NA,NA,NA,NA


In [58]:
##ride_history

ride_history = f"""

    WITH fe_tbl AS (

        SELECT
            user_id
        FROM
            pricing.fare_estimates_enriched fee
        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fee.yyyymmdd
            AND sfe.fare_estimate_id = fee.fare_estimate_id
            AND sfe.service_name = fee.service_name    
        WHERE 
            fee.yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND fee.service_detail_id = '{service_detail_id}'
            AND (session_service_level = 'All' OR session_service_level = '{service}')
        GROUP BY 1
    )

    SELECT
        user_id,
        dec_net_rides,
        CASE WHEN dec_net_rides > 0 THEN 'YES' ELSE 'NO' END AS dec_net_customers_flag,
        feb_net_rides,
        CASE WHEN feb_net_rides > 0 THEN 'YES' ELSE 'NO' END AS feb_net_customers_flag,
        dec_routeDiff_count,
        feb_routeDiff_count
    FROM
        (
        SELECT
            fe_tbl.user_id,
            COUNT(DISTINCT a.order_id) dec_net_rides,
            COUNT(DISTINCT CASE WHEN a.fare_recalculated_reason = 'routeDiff' THEN a.order_id END) dec_routeDiff_count,
            COUNT(DISTINCT b.order_id) feb_net_rides,
            COUNT(DISTINCT CASE WHEN b.fare_recalculated_reason = 'routeDiff' THEN b.order_id END) feb_routeDiff_count
        FROM 
            fe_tbl
        LEFT JOIN 
            orders.order_logs_snapshot a 
            ON fe_tbl.user_id = a.customer_id
            AND a.yyyymmdd BETWEEN '20231201' AND '20231231'
            AND a.order_status = 'dropped'
            AND a.service_obj_service_name = '{service}'
        LEFT JOIN 
            orders.order_logs_snapshot b
            ON fe_tbl.user_id = b.customer_id
            AND b.yyyymmdd BETWEEN '20240201' AND '20240229'
            AND b.order_status = 'dropped'
            AND b.service_obj_service_name = '{service}'
        GROUP BY 1
        )

"""

df_ride_history = pd.read_sql(ride_history, connection)

##Local copy
df_ride_history.to_csv(local_datasets + 'raw/ride_history_{}_{}_{}_to_{}.csv' 
                           .format(city, service, start_date, end_date), index=False 
                          )

### Reading local extract

In [73]:
df_customer_funnel = pd.read_csv(local_datasets + 'raw/customer_funnel_{}_{}_{}_to_{}.csv'
                                 .format(city, service, start_date, end_date)
                                )
df_customer_segment = pd.read_csv(local_datasets + 'raw/customer_segment_{}_{}_{}_to_{}.csv'
                                 .format(city, service, start_date, end_date)
                                )
df_ride_history = pd.read_csv(local_datasets + 'raw/ride_history_{}_{}_{}_to_{}.csv'
                                 .format(city, service, start_date, end_date)
                                )

In [74]:
print(df_customer_funnel.shape)
print(df_customer_funnel.user_id.nunique())
print(df_customer_segment.shape)
print(df_customer_segment.user_id.nunique())
print(df_ride_history.shape)
print(df_ride_history.user_id.nunique())

(606873, 10)
606873
(606873, 11)
606873
(606873, 7)
606873


In [75]:
df_customer_funnel.head(2)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,fare_shock_count,jan_routeDiff_count
0,5f43af7ad358d132cc510666,NaN,3,0,0,1,0,0,0,0
1,652aead0fdae147e7a25280c,7.023704e+09,31,3,2,19,3,2,1,1


In [76]:
df_customer_segment.head(2)

,user_id,gender,ltr_jan1,ltr_jan31,jan_net_rides,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31
0,648c29c07f3932084274f699,MALE,66.0,66.0,0.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK
1,63d0085d319e74dff8c961e1,MALE,47.0,47.0,0.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK


In [77]:
df_ride_history.head(2)

,user_id,dec_net_rides,dec_net_customers_flag,feb_net_rides,feb_net_customers_flag,dec_routeDiff_count,feb_routeDiff_count
0,63a6c089bc6a642e2b2a1717,14,YES,12,YES,10,6
1,6104c9d4a318629c3558cfda,15,YES,9,YES,2,0


In [78]:
## Joining DataFrames
df_customer_merge_raw = pd.merge(df_customer_funnel, df_customer_segment, on='user_id') \
                      .merge(df_ride_history, on='user_id')

df_customer_merge = df_customer_merge_raw.copy()
df_customer_merge.head(5)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,fare_shock_count,jan_routeDiff_count,gender,ltr_jan1,ltr_jan31,jan_net_rides,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31,dec_net_rides,dec_net_customers_flag,feb_net_rides,feb_net_customers_flag,dec_routeDiff_count,feb_routeDiff_count
0,5f43af7ad358d132cc510666,NaN,3,0,0,1,0,0,0,0,FEMALE,2.0,2.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,ONLY_AUTO,ONLY_AUTO,0,NO,0,NO,0,0
1,652aead0fdae147e7a25280c,7.023704e+09,31,3,2,19,3,2,1,1,FEMALE,14.0,14.0,0.0,HOOK,HOOK,STATIONARY,STATIONARY,LINK_AUTO,LINK_AUTO,1,YES,1,YES,0,1
2,634f9aad7b3bba469e71445e,7.597199e+09,10,2,2,4,2,2,2,2,MALE,11.0,11.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,0,NO,0,NO,0,0
3,633c71cbf5442130e2036e41,NaN,1,0,0,1,0,0,0,0,FEMALE,4.0,4.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,ONLY_AUTO,ONLY_AUTO,0,NO,0,NO,0,0
4,65a6354fdfa89c6adefdd4bc,NaN,1,0,0,1,0,0,0,0,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NO,0,NO,0,0


In [79]:
df_customer_merge['fe2rr'] = df_customer_merge['rr_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['fe2net'] = df_customer_merge['net_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['g2n'] = df_customer_merge['net_count']*100.0/df_customer_merge['rr_count']
df_customer_merge['g2n_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['gross_sessions']
df_customer_merge['fe2rr_session'] = df_customer_merge['gross_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge['fe2net_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge = df_customer_merge.round(2)

In [80]:
df_customer_merge.columns

Index(['user_id', 'customer_mobile', 'fe_count', 'rr_count', 'net_count',
       'fe_session', 'gross_sessions', 'net_sessions', 'fare_shock_count',
       'jan_routeDiff_count', 'gender', 'ltr_jan1', 'ltr_jan31',
       'jan_net_rides', 'link_lifetime_stage_jan1',
       'link_lifetime_stage_jan31', 'recency_segment_jan1',
       'recency_segment_jan31', 'service_affinity_jan1',
       'service_affinity_jan31', 'dec_net_rides', 'dec_net_customers_flag',
       'feb_net_rides', 'feb_net_customers_flag', 'dec_routeDiff_count',
       'feb_routeDiff_count', 'fe2rr', 'fe2net', 'g2n', 'g2n_session',
       'fe2rr_session', 'fe2net_session'],
      dtype='object')

In [81]:
df_customer_merge = df_customer_merge[['user_id', 'gender', 'customer_mobile',
                                       'fe_count', 'rr_count', 'net_count', 
                                       'fe2rr', 'g2n', 'fe2net',
                                       'fe_session','gross_sessions', 'net_sessions',
                                       'fe2rr_session', 'g2n_session', 'fe2net_session',
                                       'fare_shock_count', 'jan_routeDiff_count',
                                       'dec_routeDiff_count', 'feb_routeDiff_count',
                                       'ltr_jan1', 'ltr_jan31',
                                       'dec_net_rides', 'jan_net_rides', 'feb_net_rides',
                                       'dec_net_customers_flag', 'feb_net_customers_flag',
                                       'link_lifetime_stage_jan1', 'link_lifetime_stage_jan31',
                                       'recency_segment_jan1', 'recency_segment_jan31',
                                       'service_affinity_jan1', 'service_affinity_jan31'
                                      ]]

In [82]:
df_customer_merge[ df_customer_merge['user_id'].isin(['611f29a78902090a1c53e8f1'])].T

,257041
user_id,611f29a78902090a1c53e8f1
gender,FEMALE
customer_mobile,9529034344.0
fe_count,89
rr_count,33
net_count,15
fe2rr,37.08
g2n,45.45
fe2net,16.85
fe_session,33


In [83]:
print(df_customer_merge.user_id.nunique(), ' - Monthly active(fe) customers')
print(df_customer_merge[df_customer_merge['net_count'] > 0].user_id.nunique(), ' - Monthly net customers')
print(df_customer_merge[df_customer_merge['fare_shock_count'] > 0].user_id.nunique(), ' - Monthly fare-shock customers')
print(df_customer_merge[df_customer_merge['jan_routeDiff_count'] > 0].user_id.nunique(), ' - Monthly routeDiff customers')

606873  - Monthly active(fe) customers
241260  - Monthly net customers
120549  - Monthly fare-shock customers
107751  - Monthly routeDiff customers


In [84]:
## Data Filter 1 

df_customer_merge \
[df_customer_merge['net_count'] > 0]\
[df_customer_merge['service_affinity_jan31'].isin(['ONLY_LINK', 'BOTH'])].user_id.nunique()

181944

In [85]:
df_stratified_random_sampling = df_customer_merge \
                                    [df_customer_merge['net_count'] > 0]\
                                    [df_customer_merge['service_affinity_jan31']\
                                     .isin(['ONLY_LINK', 'BOTH'])]
df_stratified_random_sampling.head(5)

,user_id,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,fe2rr_session,g2n_session,fe2net_session,fare_shock_count,jan_routeDiff_count,dec_routeDiff_count,feb_routeDiff_count,ltr_jan1,ltr_jan31,dec_net_rides,jan_net_rides,feb_net_rides,dec_net_customers_flag,feb_net_customers_flag,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31
2,634f9aad7b3bba469e71445e,MALE,7.597199e+09,10,2,2,20.00,100.00,20.00,4,2,2,50.0,100.00,50.00,2,2,0,0,11.0,11.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
9,6292c5762b800fc542f7bb80,MALE,9.166244e+09,4,1,1,25.00,100.00,25.00,2,1,1,50.0,100.00,50.00,0,0,0,0,5.0,5.0,1,0.0,1,YES,YES,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
10,610950a41d7c30301ff9607b,MALE,7.297003e+09,51,22,11,43.14,50.00,21.57,16,14,11,87.5,78.57,68.75,7,7,5,2,66.0,66.0,8,0.0,4,YES,YES,COMMITTED,COMMITTED,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK
11,620b8752fad5995a72bfc430,MALE,7.976074e+09,7,4,2,57.14,50.00,28.57,5,4,2,80.0,50.00,40.00,1,0,1,1,9.0,9.0,4,0.0,1,YES,YES,SUSTENANCE,SUSTENANCE,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK
12,63b16688bb4490cb9eb02ae8,MALE,8.209388e+09,12,3,2,25.00,66.67,16.67,5,3,2,60.0,66.67,40.00,0,0,0,0,8.0,8.0,1,0.0,1,YES,YES,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK


In [86]:
df_stratified_random_sampling['customer_mobile'] = df_stratified_random_sampling['customer_mobile'].apply(str)

In [87]:
df_stratified_random_sampling.head(5)

,user_id,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,fe2rr_session,g2n_session,fe2net_session,fare_shock_count,jan_routeDiff_count,dec_routeDiff_count,feb_routeDiff_count,ltr_jan1,ltr_jan31,dec_net_rides,jan_net_rides,feb_net_rides,dec_net_customers_flag,feb_net_customers_flag,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31
2,634f9aad7b3bba469e71445e,MALE,7597199064.0,10,2,2,20.00,100.00,20.00,4,2,2,50.0,100.00,50.00,2,2,0,0,11.0,11.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
9,6292c5762b800fc542f7bb80,MALE,9166244141.0,4,1,1,25.00,100.00,25.00,2,1,1,50.0,100.00,50.00,0,0,0,0,5.0,5.0,1,0.0,1,YES,YES,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK
10,610950a41d7c30301ff9607b,MALE,7297003131.0,51,22,11,43.14,50.00,21.57,16,14,11,87.5,78.57,68.75,7,7,5,2,66.0,66.0,8,0.0,4,YES,YES,COMMITTED,COMMITTED,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK
11,620b8752fad5995a72bfc430,MALE,7976073896.0,7,4,2,57.14,50.00,28.57,5,4,2,80.0,50.00,40.00,1,0,1,1,9.0,9.0,4,0.0,1,YES,YES,SUSTENANCE,SUSTENANCE,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK
12,63b16688bb4490cb9eb02ae8,MALE,8209387987.0,12,3,2,25.00,66.67,16.67,5,3,2,60.0,66.67,40.00,0,0,0,0,8.0,8.0,1,0.0,1,YES,YES,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK


## Stratified Random Sampling

In [88]:
stratify_grouper_col = ['fe_session','gross_sessions', 'net_sessions',
                        'fe2rr_session', 'g2n_session', 'fe2net_session',
                        'dec_routeDiff_count', 'jan_routeDiff_count', 'feb_routeDiff_count',
                        'ltr_jan1', 'ltr_jan31', 'link_lifetime_stage_jan31',
                        'recency_segment_jan31', 'service_affinity_jan31'
                        ]

In [96]:
random_seed = 42

# Split the df into tc_groups
df_train, df_test = train_test_split(df_stratified_random_sampling, 
                                     test_size=0.5, 
                                     random_state=random_seed 
                                    )
#                                      stratify=df_stratified_random_sampling[stratify_grouper_col])
df_train['group_tc'] = 'control'
df_test['group_tc'] = 'test'

df_final_sample = pd.concat([df_train, df_test])

In [97]:
print(df_stratified_random_sampling.shape)
print(df_final_sample.shape)
print(df_train.shape)
print(df_test.shape)

(181944, 32)
(181944, 33)
(90972, 33)
(90972, 33)


In [98]:
df_final_sample

,user_id,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,fe2rr_session,g2n_session,fe2net_session,fare_shock_count,jan_routeDiff_count,dec_routeDiff_count,feb_routeDiff_count,ltr_jan1,ltr_jan31,dec_net_rides,jan_net_rides,feb_net_rides,dec_net_customers_flag,feb_net_customers_flag,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31,group_tc
71609,65375f0cb5145af268943dce,MALE,9529866655.0,11,5,4,45.45,80.00,36.36,6,5,4,83.33,80.00,66.67,2,2,0,2,29.0,29.0,0,0.0,12,NO,YES,HOOK,HOOK,RECENT,RECENT,ONLY_LINK,ONLY_LINK,control
116081,64b7c3e8b634456d2b537538,FEMALE,9664046813.0,23,13,10,56.52,76.92,43.48,12,11,10,91.67,90.91,83.33,2,2,0,3,17.0,17.0,2,0.0,6,YES,YES,CHURN_OTB,CHURN_OTB,RECENT,RECENT,ONLY_LINK,ONLY_LINK,control
508343,63e2586d99ac6424766bb0f5,FEMALE,8094071441.0,11,5,2,45.45,40.00,18.18,3,3,2,100.00,66.67,66.67,1,1,0,0,8.0,8.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
521205,64d250793a4db9a40d800ad0,MALE,9376826210.0,2,1,1,50.00,100.00,50.00,1,1,1,100.00,100.00,100.00,0,0,0,0,1.0,1.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
353989,65ad60287f70ea61d0c5fefe,MALE,9772959229.0,15,3,2,20.00,66.67,13.33,8,3,2,37.50,66.67,25.00,0,0,0,0,2.0,2.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176633,641a98cca609402274348700,MALE,9351503576.0,14,3,3,21.43,100.00,21.43,7,3,3,42.86,100.00,42.86,1,1,0,0,3.0,3.0,0,0.0,0,NO,NO,HANDHOLDING,HANDHOLDING,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK,test
601959,65a5e23db3bae63168d591ed,FEMALE,9868799477.0,8,3,3,37.50,100.00,37.50,6,3,3,50.00,100.00,50.00,1,1,0,0,3.0,3.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test
518588,5d356fa0f8fc6575c089eef5,MALE,9602414254.0,3,1,1,33.33,100.00,33.33,1,1,1,100.00,100.00,100.00,1,0,0,0,1.0,1.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test
287592,5f718b9b207b1e410246fbcb,MALE,9799350399.0,2,1,1,50.00,100.00,50.00,2,1,1,50.00,100.00,50.00,0,0,0,0,22.0,22.0,2,0.0,0,YES,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test


### Balance Groups

In [99]:
df_final_sample.groupby(['group_tc'])\
.agg(user_base = pd.NamedAgg('user_id', 'nunique'),
     fe_count = pd.NamedAgg('fe_count', 'sum'),
     rr_count = pd.NamedAgg('rr_count', 'sum'),
     net_count = pd.NamedAgg('net_count', 'sum'),
     fe_session = pd.NamedAgg('fe_session', 'sum'),
     gross_sessions = pd.NamedAgg('gross_sessions', 'sum'),
     net_sessions = pd.NamedAgg('net_sessions', 'sum'),
     dec_routeDiff_count = pd.NamedAgg('dec_routeDiff_count', 'sum'),
     jan_routeDiff_count = pd.NamedAgg('jan_routeDiff_count', 'sum'),
     feb_routeDiff_count = pd.NamedAgg('feb_routeDiff_count', 'sum'),
     lifetime_stage_dist = pd.NamedAgg(column='link_lifetime_stage_jan31', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),   
     recency_segment_dist = pd.NamedAgg(column='recency_segment_jan1', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),     
     service_affinity_dist = pd.NamedAgg(column='service_affinity_jan31', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
     dec_net_customers_flag_distr = pd.NamedAgg(column='dec_net_customers_flag', 
                                                aggfunc=lambda x: x.value_counts().to_dict()),
     feb_net_customers_flag_distr = pd.NamedAgg(column='feb_net_customers_flag', 
                                                aggfunc=lambda x: x.value_counts().to_dict()),
     mean_ltr_jan1 = pd.NamedAgg('ltr_jan1', 'median'),
     mean_ltr_jan31 = pd.NamedAgg('ltr_jan31', 'median')     
    ).T

group_tc,control,test
user_base,90972,90972
fe_count,1039757,1033144
rr_count,361713,358816
net_count,219846,217324
fe_session,453191,449035
gross_sessions,290811,288563
net_sessions,219837,217310
dec_routeDiff_count,35120,34590
jan_routeDiff_count,61677,60994
feb_routeDiff_count,35963,34728


In [100]:
##Local extract

df_train[['user_id','customer_mobile','group_tc']].to_csv(local_datasets + 'processed/df_control_{}_{}_price_buffer_exp_selector.csv'\
                       .format(city, service), 
                       index=False )

##Local extract
df_test[['user_id','customer_mobile','group_tc']].to_csv(local_datasets + 'processed/df_test_{}_{}_price_buffer_exp_selector.csv'\
                       .format(city, service), 
                       index=False )

In [101]:
common_values = df_train[df_train['user_id'].isin(df_test['user_id'])]['user_id'].tolist()

if common_values:
    print(f"Common values in user_id: {common_values}")
else:
    print("No common values found.")

No common values found.


## User Selector for Experiment

In [102]:
##Local extract

df_final_sample.to_csv(local_datasets + 'processed/{}_{}_price_buffer_exp_group_tc.csv'\
                       .format(city, service), 
                       index=False )

In [103]:
df_final_group_tc = pd.read_csv(local_datasets + 'processed/{}_{}_price_buffer_exp_group_tc.csv'\
                       .format(city, service)
                               )

In [104]:
df_final_group_tc

,user_id,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,fe2rr_session,g2n_session,fe2net_session,fare_shock_count,jan_routeDiff_count,dec_routeDiff_count,feb_routeDiff_count,ltr_jan1,ltr_jan31,dec_net_rides,jan_net_rides,feb_net_rides,dec_net_customers_flag,feb_net_customers_flag,link_lifetime_stage_jan1,link_lifetime_stage_jan31,recency_segment_jan1,recency_segment_jan31,service_affinity_jan1,service_affinity_jan31,group_tc
0,65375f0cb5145af268943dce,MALE,9.529867e+09,11,5,4,45.45,80.00,36.36,6,5,4,83.33,80.00,66.67,2,2,0,2,29.0,29.0,0,0.0,12,NO,YES,HOOK,HOOK,RECENT,RECENT,ONLY_LINK,ONLY_LINK,control
1,64b7c3e8b634456d2b537538,FEMALE,9.664047e+09,23,13,10,56.52,76.92,43.48,12,11,10,91.67,90.91,83.33,2,2,0,3,17.0,17.0,2,0.0,6,YES,YES,CHURN_OTB,CHURN_OTB,RECENT,RECENT,ONLY_LINK,ONLY_LINK,control
2,63e2586d99ac6424766bb0f5,FEMALE,8.094071e+09,11,5,2,45.45,40.00,18.18,3,3,2,100.00,66.67,66.67,1,1,0,0,8.0,8.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
3,64d250793a4db9a40d800ad0,MALE,9.376826e+09,2,1,1,50.00,100.00,50.00,1,1,1,100.00,100.00,100.00,0,0,0,0,1.0,1.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
4,65ad60287f70ea61d0c5fefe,MALE,9.772959e+09,15,3,2,20.00,66.67,13.33,8,3,2,37.50,66.67,25.00,0,0,0,0,2.0,2.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181939,641a98cca609402274348700,MALE,9.351504e+09,14,3,3,21.43,100.00,21.43,7,3,3,42.86,100.00,42.86,1,1,0,0,3.0,3.0,0,0.0,0,NO,NO,HANDHOLDING,HANDHOLDING,STATIONARY,STATIONARY,ONLY_LINK,ONLY_LINK,test
181940,65a5e23db3bae63168d591ed,FEMALE,9.868799e+09,8,3,3,37.50,100.00,37.50,6,3,3,50.00,100.00,50.00,1,1,0,0,3.0,3.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test
181941,5d356fa0f8fc6575c089eef5,MALE,9.602414e+09,3,1,1,33.33,100.00,33.33,1,1,1,100.00,100.00,100.00,1,0,0,0,1.0,1.0,0,0.0,0,NO,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test
181942,5f718b9b207b1e410246fbcb,MALE,9.799350e+09,2,1,1,50.00,100.00,50.00,2,1,1,50.00,100.00,50.00,0,0,0,0,22.0,22.0,2,0.0,0,YES,NO,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_LINK,ONLY_LINK,test


In [48]:
df_final_group_tc[['user_id','customer_mobile','group_tc']]\
.to_csv(local_datasets + 'processed/customer_session_{}_{}_price_buffer_exp_group_tc.csv'\
.format(city, service), index=False )

In [106]:
df_final_group_tc[['customer_mobile']]\
.to_csv(local_datasets + 'processed/final_customer_session_{}_{}_price_buffer_exp_group.csv'\
.format(city, service), index=False )

In [105]:
local_datasets

'/Users/rapido/local-datasets/customer/fare-shock/price_buffer_exp/'

In [107]:
df_final_group_tc[['user_id']]

,user_id
0,65375f0cb5145af268943dce
1,64b7c3e8b634456d2b537538
2,63e2586d99ac6424766bb0f5
3,64d250793a4db9a40d800ad0
4,65ad60287f70ea61d0c5fefe
...,...
181939,641a98cca609402274348700
181940,65a5e23db3bae63168d591ed
181941,5d356fa0f8fc6575c089eef5
181942,5f718b9b207b1e410246fbcb


In [108]:
df_final_group_tc[['user_id']]\
.to_csv(local_datasets + 'processed/JAIPUR_LINK_PRICE_BUFF_SELECTOR_EXP_GROUP.csv'\
.format(city, service), index=False )